# Maintenance Classification con Polars

Versión transformada del flujo de pandas a **Polars**.  
Nota: para `scikit-learn`, `seaborn` y algunos gráficos se convierte a pandas únicamente en la frontera de integración, porque esas librerías trabajan mejor con `DataFrame` de pandas o arreglos NumPy.

In [ ]:
import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import graphviz

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn import svm, tree
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

from xgboost import XGBClassifier
from scipy.stats import randint, uniform, loguniform

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sqlite3
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

## 1. Carga de datos con Polars

In [ ]:
data = pl.read_csv("../data/ai4i2020.csv")

print(data.shape)
data.head(10)

In [ ]:
# Filtrar registros con falla de máquina
data.filter(pl.col("Machine failure") == 1)

In [ ]:
# Conteo de clases
data.group_by("Machine failure").len().sort("Machine failure")

In [ ]:
# Resumen estadístico
data.describe()

In [ ]:
# Tipos de datos
data.schema

In [ ]:
cols = [c for c in data.columns if c not in ["UDI", "Product ID"]]
print(cols)

unique_values = {
    col: data.select(pl.col(col).unique()).to_series().to_list()
    for col in ["Type", "Machine failure", "Air temperature [K]"]
}

unique_values

In [ ]:
# Valores únicos para todas las columnas, en formato largo
unique_values_data = pl.concat(
    [
        data.select(pl.col(col).unique().alias("value"))
            .with_columns(pl.lit(col).alias("column"))
            .select(["column", "value"])
        for col in cols
    ],
    how="vertical"
)

unique_values_data.head(32)

## 2. Limpieza básica

In [ ]:
# Conteo de nulos por columna
data.null_count()

In [ ]:
# Eliminar nulos
df = data.drop_nulls()
df.null_count()

In [ ]:
# Columnas disponibles
data.columns

In [ ]:
# Eliminar columnas identificadoras
df = data.drop(["UDI", "Product ID"])
df.columns

In [ ]:
df.shape

In [ ]:
# Revisar y eliminar duplicados
duplicated_features = df.is_duplicated().sum()
print("Number of duplicates ----->>>", duplicated_features)

df = df.unique(maintain_order=True)

duplicated_features = df.is_duplicated().sum()
print("Number of duplicates after cleaning ----->>>", duplicated_features)

## 3. Transformación de unidades y nombres

In [ ]:
df = (
    df
    .with_columns([
        (pl.col("Air temperature [K]") - 272.15).alias("Air temperature [°C]"),
        (pl.col("Process temperature [K]") - 272.15).alias("Process temperature [°C]")
    ])
    .drop(["Air temperature [K]", "Process temperature [K]"])
)

print(df.columns)
df.head()

In [ ]:
df = df.rename({
    "Air temperature [°C]": "Air_temperature",
    "Process temperature [°C]": "Process_temperature",
    "Rotational speed [rpm]": "Rotational_speed",
    "Torque [Nm]": "Torque",
    "Tool wear [min]": "Tool_wear",
    "Failure Type": "Failure_Type"
})

df.head(10)

## 4. Análisis exploratorio

In [ ]:
# Matriz de correlación
numeric_cols = [name for name, dtype in df.schema.items() if dtype.is_numeric()]

f1 = {"family": "DejaVu Sans", "size": 35, "color": "r"}

plt.figure(figsize=(30, 30), dpi=120)

sns.heatmap(
    df.select(numeric_cols).to_pandas().corr(),
    annot=True,
    fmt="0.2f",
    cmap="viridis",
    linewidths=0.5
)

plt.xticks(rotation=45, color="r", fontsize=25)
plt.yticks(rotation=-45, color="r", fontsize=25)
plt.xlabel("Características", fontdict=f1)
plt.ylabel("Características", fontdict=f1)
plt.title("Matriz de Correlación - Heatmap", fontdict=f1)
plt.show()

In [ ]:
# Detección de outliers con IQR
features = [name for name, dtype in df.schema.items() if dtype.is_numeric()]

for col in features:
    q1, q3 = df.select(pl.col(col).quantile(0.25), pl.col(col).quantile(0.75)).row(0)
    iqr = q3 - q1
    low_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outliers = (
        df
        .filter((pl.col(col) > upper_limit) | (pl.col(col) < low_limit))
        .select(col)
        .to_series()
        .to_list()
    )

    if len(outliers) == 0:
        print(f" * -- >> there No outlier in {col} feature")
    else:
        print(f"There are outliers in this feature {col}")

    print(
        f"Q1 of {col} --->>> {q1}\n"
        f"Q3 of {col} ---->>> {q3}\n"
        f"IQR --->>> {iqr}\n"
        f"low_limit --->>> {low_limit}\n"
        f"upper_limit --->>> {upper_limit}\n"
        f"outliers ---->>> {outliers}\n"
        f"Number of outliers --->>> {len(outliers)}"
    )
    print("-" * 90)

In [ ]:
# Boxplot requiere pandas/seaborn
plt.figure(figsize=(20, 20), dpi=250)
plt.title("This representation checks outliers")
plt.xlabel("Features")
plt.ylabel("Count")
plt.xticks(rotation=45, color="b", fontsize=25)
sns.boxplot(data=df.select(features).to_pandas())
plt.show()

In [ ]:
# Renombrar target
df = df.rename({"Machine failure": "Target"})
print(df.columns)

## 5. Manejo de outliers conservando índice explícito

In [ ]:
# En Polars no existe índice implícito como en pandas.
# Para replicar df.index.difference(...), creamos un índice explícito con with_row_index.

df_indexed = df.with_row_index("row_id")
x_class = df_indexed.drop("Target")
y_class = df_indexed.select(["row_id", "Target"])

In [ ]:
def remove_outliers_iqr_polars(df_pl: pl.DataFrame, index_col: str = "row_id"):
    outlier_counts = {}
    outlier_means = {}
    df_cleaned = df_pl

    numeric_columns = [
        name for name, dtype in df_pl.schema.items()
        if dtype.is_numeric() and name != index_col
    ]

    for column in numeric_columns:
        q1, q3 = df_cleaned.select(
            pl.col(column).quantile(0.25),
            pl.col(column).quantile(0.75)
        ).row(0)

        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outliers = df_cleaned.filter(
            (pl.col(column) < lower_bound) | (pl.col(column) > upper_bound)
        )

        outlier_counts[column] = outliers.height

        if outliers.height > 0:
            outlier_means[column] = outliers.select(pl.col(column).mean()).item()
        else:
            outlier_means[column] = 0.0

        df_cleaned = df_cleaned.filter(
            (pl.col(column) >= lower_bound) & (pl.col(column) <= upper_bound)
        )

    return df_cleaned, outlier_counts, outlier_means


df_cleaned_without_target, outlier_counts, outlier_means = remove_outliers_iqr_polars(x_class)

print("Number of outliers detected in each column:")
print(outlier_counts)
print("-" * 70)

print("\nMean of outliers in each column:")
print(outlier_means)
print("-" * 70)

print("\nDataFrame after removing outliers:")
print(df_cleaned_without_target)
print("Tamaño de la tabla eliminando los outliers")
print(df_cleaned_without_target.shape)

In [ ]:
# Equivalente a: df.index.difference(df_cleaned_without_target.index)
indices_solo_df = (
    df_indexed
    .select("row_id")
    .join(df_cleaned_without_target.select("row_id"), on="row_id", how="anti")
)

indices_solo_df

In [ ]:
# Equivalente a: df.loc[df_cleaned_without_target_no_in_df]
df_indexed.join(indices_solo_df, on="row_id", how="inner")

In [ ]:
def clip_outliers_iqr_polars(df_pl: pl.DataFrame, columns: list[str]) -> pl.DataFrame:
    result = df_pl

    for column in columns:
        q1, q3 = result.select(
            pl.col(column).quantile(0.25),
            pl.col(column).quantile(0.75)
        ).row(0)

        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        result = result.with_columns(
            pl.col(column).clip(lower_bound, upper_bound).alias(column)
        )

    return result


df_cleaned_without_target = clip_outliers_iqr_polars(
    df_cleaned_without_target,
    columns=["Rotational_speed", "Torque"]
)

print("Data after cleaned data and clipping outliers:", df_cleaned_without_target.shape)
print("Original data shape:", df.shape)

## 6. Visualizaciones con conversión puntual a pandas

In [ ]:
color_palette = ["#00FF00", "#DFFF00", "#228B22", "#808000", "#2E8B57", "#008080"]

type_avg = (
    df
    .group_by("Type")
    .agg(pl.col("Air_temperature").mean().alias("Air_temperature"))
    .sort("Air_temperature", descending=True)
)

print("Rates of Air_temperature at every type:")
print(type_avg)

type_index = type_avg.to_pandas()
df_pd = df.to_pandas()

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "pie"}, {"type": "bar"}]])

fig.add_trace(
    go.Pie(labels=type_index["Type"], values=type_index["Air_temperature"], marker=dict(colors=color_palette)),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(name="Air_temperature", x=df_pd["Type"], y=df_pd["Air_temperature"], marker_color="#7CFC00"),
    row=1,
    col=2
)

fig.update_layout(
    title="Observation Type vs Air_temperature",
    legend_title="Types & Air_temperature",
    width=1200,
    height=600,
    showlegend=True
)

fig.show()

In [ ]:
# Histogramas con seaborn
df_pd = df.to_pandas()

for col in df.columns:
    sns.displot(df_pd, x=col, kde=True, bins=100, color="red", facecolor="yellow", height=5, aspect=3.5)

In [ ]:
# Scatter con Plotly
fig = px.scatter(
    df.to_pandas(),
    x="Air_temperature",
    y="Rotational_speed",
    color="Rotational_speed"
)

fig.update_layout(
    title="Air_temperature vs Rotational_speed",
    xaxis_title="Air_temperature",
    yaxis_title="Rotational_speed"
)

fig.show()

In [ ]:
fig = px.histogram(df.to_pandas(), x="Air_temperature", color="Type")
fig.update_layout(
    bargap=0.2,
    title="Observation Air_temperature vs Type",
    legend_title="Type",
    width=800,
    height=600
)
fig.show()

In [ ]:
type_torque = (
    df
    .group_by("Type")
    .agg(pl.col("Torque").mean().alias("Torque"))
    .sort("Torque", descending=True)
)

print("Rates of Torque at every type:")
print(type_torque)

type_index = type_torque.to_pandas()
df_pd = df.to_pandas()

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "pie"}, {"type": "bar"}]])

fig.add_trace(
    go.Pie(labels=type_index["Type"], values=type_index["Torque"], marker=dict(colors=color_palette)),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(name="Torque", x=df_pd["Type"], y=df_pd["Torque"], marker_color="#239b56"),
    row=1,
    col=2
)

fig.update_layout(
    title="Observation Type vs Torque",
    legend_title="Types & Torque",
    width=1200,
    height=600,
    showlegend=True
)

fig.show()

In [ ]:
df.select([name for name, dtype in df.schema.items() if dtype.is_numeric()]).to_pandas().hist(figsize=(25, 25))
plt.show()

## 7. Preparación para Machine Learning

In [ ]:
# One-hot encoding con Polars.
# Esto reemplaza el LabelEncoder aplicado a columnas categóricas.
categorical_cols = [name for name, dtype in df.schema.items() if dtype == pl.String]

df_encoded = df.to_dummies(columns=categorical_cols)

df_encoded.head()

In [ ]:
# Escalamiento con MinMaxScaler.
# sklearn devuelve NumPy; reconstruimos Polars al final.
numerical_features = df_encoded.select([
    name for name, dtype in df_encoded.schema.items()
    if dtype.is_numeric()
])

scaler = MinMaxScaler()
scaled_numerical_features = scaler.fit_transform(numerical_features.to_numpy())

scaled_numerical_df = pl.DataFrame(
    scaled_numerical_features,
    schema=numerical_features.columns
)

scaled_numerical_df.head()

In [ ]:
# Separar X e y
drop_cols = [
    "Target",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
]

# Si se codificó Failure_Type con dummies, se elimina para evitar fuga de información.
drop_cols += [c for c in scaled_numerical_df.columns if c.startswith("Failure_Type")]

drop_cols = [c for c in drop_cols if c in scaled_numerical_df.columns]

x_class_pl = scaled_numerical_df.drop(drop_cols)
y_class_pl = scaled_numerical_df.select("Target")

print(x_class_pl.columns)
print(y_class_pl.head())

In [ ]:
# Conversión puntual para sklearn
x_class = x_class_pl.to_pandas()
y_class = y_class_pl.to_series().to_pandas()

x_train, x_test, y_train, y_test = train_test_split(
    x_class,
    y_class,
    test_size=0.3,
    random_state=42,
    stratify=y_class
)

print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 8. Modelos y búsqueda de hiperparámetros

In [ ]:
cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

randomsearch_rf = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ),
    param_distributions={
        "n_estimators": randint(100, 500),
        "max_depth": [3, 5, 7, 10, 15, 20, None],
        "min_samples_split": randint(5, 50),
        "min_samples_leaf": randint(2, 30),
        "max_features": ["sqrt", "log2", None],
        "bootstrap": [True, False]
    },
    n_iter=50,
    cv=cv_strategy,
    scoring="f1",
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)

randomsearch_gb = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(
        random_state=42
    ),
    param_distributions={
        "n_estimators": randint(50, 300),
        "learning_rate": loguniform(0.01, 0.2),
        "max_depth": randint(2, 6),
        "min_samples_split": randint(5, 50),
        "min_samples_leaf": randint(2, 30),
        "subsample": uniform(0.6, 0.4),
        "max_features": ["sqrt", "log2", None]
    },
    n_iter=50,
    cv=cv_strategy,
    scoring="f1",
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)

scale_pos_weight_value = y_train.value_counts().loc[0] / y_train.value_counts().loc[1]

randomsearch_xgb = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    ),
    param_distributions={
        "n_estimators": randint(100, 500),
        "max_depth": randint(2, 7),
        "learning_rate": loguniform(0.005, 0.2),
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.6, 0.4),
        "min_child_weight": randint(1, 15),
        "gamma": uniform(0, 1),
        "reg_alpha": loguniform(1e-4, 1),
        "reg_lambda": loguniform(0.5, 10),
        "scale_pos_weight": [
            scale_pos_weight_value * 0.5,
            scale_pos_weight_value,
            scale_pos_weight_value * 1.5,
            scale_pos_weight_value * 2
        ]
    },
    n_iter=100,
    cv=cv_strategy,
    scoring="average_precision",
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)

models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),
    "DecisionTreeClassifier": DecisionTreeClassifier(
        max_depth=5,
        max_features=6,
        class_weight="balanced",
        random_state=42
    ),
    "RandomForestClassifier": randomsearch_rf,
    "svm": svm.SVC(
        random_state=42,
        kernel="poly",
        class_weight="balanced",
        probability=True
    ),
    "GradientBoostingClassifier": randomsearch_gb,
    "XGBoostClassifier": randomsearch_xgb
}

In [ ]:
for model_name, model in models.items():
    print("=" * 80)
    print(f"Entrenando modelo: {model_name}")

    model.fit(x_train, y_train)

    if hasattr(model, "best_params_"):
        print("Mejores parámetros:")
        print(model.best_params_)

    if hasattr(model, "best_score_"):
        print(f"Mejor score CV: {model.best_score_:.4f}")

    print("=" * 80)

## 9. Evaluación de modelos

In [ ]:
results = []

for model_name, model in models.items():
    print("=" * 80)
    print(f"Evaluando modelo: {model_name}")

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    if hasattr(model, "predict_proba"):
        y_train_proba = model.predict_proba(x_train)[:, 1]
        y_test_proba = model.predict_proba(x_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_train_proba = model.decision_function(x_train)
        y_test_proba = model.decision_function(x_test)
    else:
        y_train_proba = None
        y_test_proba = None

    acc_train = accuracy_score(y_train, y_train_pred)
    balanced_acc_train = balanced_accuracy_score(y_train, y_train_pred)
    precision_train = precision_score(y_train, y_train_pred, zero_division=0)
    recall_train = recall_score(y_train, y_train_pred, zero_division=0)
    f1_train = f1_score(y_train, y_train_pred, zero_division=0)

    acc_test = accuracy_score(y_test, y_test_pred)
    balanced_acc_test = balanced_accuracy_score(y_test, y_test_pred)
    precision_test = precision_score(y_test, y_test_pred, zero_division=0)
    recall_test = recall_score(y_test, y_test_pred, zero_division=0)
    f1_test = f1_score(y_test, y_test_pred, zero_division=0)

    roc_auc_train = roc_auc_score(y_train, y_train_proba) if y_train_proba is not None else np.nan
    avg_precision_train = average_precision_score(y_train, y_train_proba) if y_train_proba is not None else np.nan

    roc_auc_test = roc_auc_score(y_test, y_test_proba) if y_test_proba is not None else np.nan
    avg_precision_test = average_precision_score(y_test, y_test_proba) if y_test_proba is not None else np.nan

    cm = confusion_matrix(y_test, y_test_pred)

    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Predicho 0", "Predicho 1"],
        yticklabels=["Real 0", "Real 1"]
    )
    plt.xlabel(f"Predicciones del modelo: {model_name}")
    plt.ylabel("Valores reales")
    plt.title(f"Matriz de confusión - {model_name}")
    plt.show()

    print("\nClassification Report TEST:")
    print(classification_report(y_test, y_test_pred, zero_division=0))

    results.append({
        "Modelo": model_name,
        "Accuracy Train": acc_train,
        "Accuracy Test": acc_test,
        "Balanced Accuracy Train": balanced_acc_train,
        "Balanced Accuracy Test": balanced_acc_test,
        "Precision Train": precision_train,
        "Precision Test": precision_test,
        "Recall Train": recall_train,
        "Recall Test": recall_test,
        "F1 Train": f1_train,
        "F1 Test": f1_test,
        "ROC-AUC Train": roc_auc_train,
        "ROC-AUC Test": roc_auc_test,
        "Average Precision Train": avg_precision_train,
        "Average Precision Test": avg_precision_test
    })

results_df = pl.DataFrame(results).sort("Balanced Accuracy Test", descending=True)

results_df

## 10. Importancia de variables

In [ ]:
decision_tree_model = models["DecisionTreeClassifier"]

plt.figure(figsize=(95, 85), dpi=150)
tree.plot_tree(
    decision_tree_model,
    filled=True,
    feature_names=x_class.columns,
    node_ids=True,
    fontsize=42
)
plt.title("Decision Tree")
plt.show()

In [ ]:
feature_importances1 = decision_tree_model.feature_importances_

importance_df = (
    pl.DataFrame({
        "Feature": list(x_class.columns),
        "Importance": feature_importances1
    })
    .sort("Importance", descending=True)
)

importance_df

In [ ]:
best_estimator_rf = models["RandomForestClassifier"].best_estimator_

importance_rf = (
    pl.DataFrame({
        "Feature": list(x_class.columns),
        "Importance": best_estimator_rf.feature_importances_
    })
    .sort("Importance", descending=True)
)

importance_rf

In [ ]:
best_estimator_gb = models["GradientBoostingClassifier"].best_estimator_

importance_gb = (
    pl.DataFrame({
        "Feature": list(x_class.columns),
        "Importance": best_estimator_gb.feature_importances_
    })
    .sort("Importance", descending=True)
)

importance_gb

## 11. Consulta de base SQLite con Polars

In [ ]:
DB_PATH = "/home/alfonso/Maintanance_prediction_API/predictions.db"

conn = sqlite3.connect(DB_PATH)

tables = pl.read_database(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print(tables)

history_df = pl.read_database(
    "SELECT * FROM predictions ORDER BY id DESC",
    conn
)

conn.close()

history_df